
# Numerical Algorithms — Tutorial + Assignment  
## Gradient Checking a Two-Layer MLP with Finite Differences (Forward, Central, Higher-Order)

This notebook is designed to run **as-is on Google Colab**.

### What you will learn
1. How the **chain rule** leads to **backpropagation** for a two-layer multilayer perceptron (MLP).
2. How to compute **analytical gradients** efficiently using backpropagation.
3. How to implement **finite-difference numerical derivatives**:
   - Forward difference (1st order)
   - Central difference (2nd order)
   - Higher-order central differences (4th and 6th order)
4. How to compare numerical gradients against backprop gradients and report errors in a table.

### What you will submit
- Completed TODOs in this notebook
- A short written report (within this notebook) including tables and discussion

---

### Historical context (brief)
- The **perceptron** was introduced by Frank Rosenblatt (1957–1958) as an early model of a neuron.
- **Multi-layer perceptrons (MLPs)** became practical when **backpropagation** (chain rule + dynamic programming) was popularized in the 1980s (notably Rumelhart, Hinton, Williams, 1986).
- Today, the same core idea—**backpropagation through computational graphs**—is used to train modern systems in:
  - language (LLMs),
  - vision (CNNs, ViTs),
  - speech (RNNs, Transformers),
  - diffusion models and many other architectures.

Backprop is fundamental because it computes exact gradients efficiently, whereas finite differences are primarily used for **debugging and gradient checking**.

---


In [ ]:

# Part 0 — Setup (Colab-ready)
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)

def set_seed(seed=0):
    np.random.seed(seed)

set_seed(42)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)



## Part 1 — Two-layer MLP and backpropagation (chain rule)

We use a two-layer MLP (one hidden layer). For a batch of \( N \) inputs:
- \( X \in \mathbb{R}^{N \times D} \)
- hidden width \( H \)
- output dimension \( K \)

---

### Forward pass

We define:

$$
Z^{(1)} = X W^{(1)} + \mathbf{1}\,(b^{(1)})^\top
$$

$$
A^{(1)} = \tanh\!\left(Z^{(1)}\right)
$$

$$
Z^{(2)} = A^{(1)} W^{(2)} + \mathbf{1}\,(b^{(2)})^\top
$$

$$
\hat{Y} = Z^{(2)}
$$

The output layer is **linear** (no activation).

---

### Loss (mean squared error)

$$
\mathcal{L} = \frac{1}{2N}\left\|\hat{Y} - Y\right\|_F^2
$$

---

### Backpropagation (key gradients)

Define the output-layer gradient:

$$
G^{(2)} := \frac{\partial \mathcal{L}}{\partial Z^{(2)}} = \frac{1}{N}(\hat{Y}-Y)
$$

Then:

$$
\frac{\partial \mathcal{L}}{\partial W^{(2)}} = (A^{(1)})^\top G^{(2)}
$$

$$
\frac{\partial \mathcal{L}}{\partial b^{(2)}} = \sum_{n=1}^{N} G^{(2)}_{n,:}
$$

Propagate to hidden:

$$
\frac{\partial \mathcal{L}}{\partial A^{(1)}} = G^{(2)} (W^{(2)})^\top
$$

Using:

$$
\frac{d}{dz}\tanh(z)=1-\tanh^2(z)
$$

Define:

$$
G^{(1)} := \frac{\partial \mathcal{L}}{\partial Z^{(1)}} =
\left(\frac{\partial \mathcal{L}}{\partial A^{(1)}}\right) \odot \left(1-\tanh^2(Z^{(1)})\right)
$$

Finally:

$$
\frac{\partial \mathcal{L}}{\partial W^{(1)}} = X^\top G^{(1)}
$$

$$
\frac{\partial \mathcal{L}}{\partial b^{(1)}} = \sum_{n=1}^{N} G^{(1)}_{n,:}
$$

---

**Important clarification.** Backpropagation does **not** use finite differences.  
It computes gradients **analytically** via the chain rule, efficiently reusing intermediate computations.



## Part 2 — Implementation: two-layer MLP (provided)

In this assignment, the MLP forward pass and backpropagation are **provided** so you can focus on numerical differentiation methods.

We implement:
- Parameter initialization
- Forward pass
- Loss computation
- Backpropagation gradients

You will later compare these backprop gradients against finite-difference gradients.


In [ ]:

# Two-layer MLP implementation (NumPy)

def init_mlp(D, H, K, scale=0.1):
    params = {
        "W1": scale * np.random.randn(D, H),
        "b1": np.zeros((H,)),
        "W2": scale * np.random.randn(H, K),
        "b2": np.zeros((K,)),
    }
    return params

def forward_mlp(X, params):
    W1, b1 = params["W1"], params["b1"]
    W2, b2 = params["W2"], params["b2"]
    
    Z1 = X @ W1 + b1[None, :]
    A1 = np.tanh(Z1)
    Z2 = A1 @ W2 + b2[None, :]
    Yhat = Z2  # linear output
    
    cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "Yhat": Yhat}
    return Yhat, cache

def mse_loss(Yhat, Y):
    N = Y.shape[0]
    diff = Yhat - Y
    loss = 0.5 * np.sum(diff * diff) / N
    return loss

def backward_mlp(Y, params, cache):
    """
    Returns gradients dict with keys: dW1, db1, dW2, db2
    """
    X, A1, Yhat = cache["X"], cache["A1"], cache["Yhat"]
    W2 = params["W2"]
    N = Y.shape[0]
    
    # G2 = dL/dZ2
    G2 = (Yhat - Y) / N  # shape (N, K)
    
    dW2 = A1.T @ G2
    db2 = np.sum(G2, axis=0)
    
    dA1 = G2 @ W2.T
    # tanh'(Z1) = 1 - tanh(Z1)^2 = 1 - A1^2
    G1 = dA1 * (1.0 - A1 * A1)
    
    dW1 = X.T @ G1
    db1 = np.sum(G1, axis=0)
    
    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

# Quick sanity check
D, H, K = 5, 4, 3
N = 8
X = np.random.randn(N, D)
Y = np.random.randn(N, K)
params = init_mlp(D, H, K)

Yhat, cache = forward_mlp(X, params)
loss = mse_loss(Yhat, Y)
grads = backward_mlp(Y, params, cache)

print("Loss:", loss)
for k, v in grads.items():
    print(k, v.shape)



## Part 3 — Numerical differentiation (assignment)

We will approximate gradients of a scalar function \( f(\theta) \) using finite differences, and compare with backpropagation gradients.

Let \( f:\mathbb{R}^m \to \mathbb{R} \). For the \( i \)-th coordinate unit vector \( e_i \), define:

### Forward difference (1st order)

$$
\frac{\partial f}{\partial \theta_i} \approx \frac{f(\theta + h e_i) - f(\theta)}{h}
$$

### Central difference (2nd order)

$$
\frac{\partial f}{\partial \theta_i} \approx \frac{f(\theta + h e_i) - f(\theta - h e_i)}{2h}
$$

### 4th-order central difference

$$
\frac{\partial f}{\partial \theta_i} \approx
\frac{-f(\theta+2he_i) + 8f(\theta+he_i) - 8f(\theta-he_i) + f(\theta-2he_i)}{12h}
$$

### 6th-order central difference

$$
\frac{\partial f}{\partial \theta_i} \approx
\frac{f(\theta-3he_i) - 9f(\theta-2he_i) + 45f(\theta-he_i) - 45f(\theta+he_i) + 9f(\theta+2he_i) - f(\theta+3he_i)}{60h}
$$

### Your task
You will implement these derivative formulas and use them to compute numerical gradients for all MLP parameters.

---



### Practical note on step size \( h \)

In finite differences, \( h \) cannot be too large (truncation error) or too small (floating-point cancellation).  
A typical range to sweep is:

$$
h \in \{10^{-1}, 10^{-2}, 10^{-3}, 10^{-4}, 10^{-5}\}
$$

You will run gradient checks across multiple \( h \) values and report relative error norms.


In [ ]:

# Utilities: pack/unpack model parameters into a flat vector for finite differences

def pack_params(params):
    """
    Flatten params into a 1D vector theta, plus metadata to unpack later.
    """
    theta_parts = []
    meta = []
    for name in ["W1", "b1", "W2", "b2"]:
        arr = params[name]
        meta.append((name, arr.shape, arr.size))
        theta_parts.append(arr.reshape(-1))
    theta = np.concatenate(theta_parts, axis=0)
    return theta, meta

def unpack_params(theta, meta):
    """
    Reconstruct params dict from flat theta using meta.
    """
    params = {}
    idx = 0
    for name, shape, size in meta:
        params[name] = theta[idx:idx+size].reshape(shape)
        idx += size
    return params

def pack_grads(grads):
    """
    Flatten backprop grads to match pack_params ordering.
    """
    parts = [
        grads["dW1"].reshape(-1),
        grads["db1"].reshape(-1),
        grads["dW2"].reshape(-1),
        grads["db2"].reshape(-1),
    ]
    return np.concatenate(parts, axis=0)

def loss_from_theta(theta, meta, X, Y):
    """
    Scalar objective f(theta): MSE loss of the MLP for parameters given by theta.
    """
    params = unpack_params(theta, meta)
    Yhat, _ = forward_mlp(X, params)
    return mse_loss(Yhat, Y)

# Prepare a small instance (kept small so FD is feasible)
set_seed(123)
D, H, K = 6, 5, 4
N = 10
X = np.random.randn(N, D)
Y = np.random.randn(N, K)
params0 = init_mlp(D, H, K, scale=0.1)

theta0, meta = pack_params(params0)
print("Total parameters m =", theta0.size)



## Part 4 — Implement finite-difference formulas (TODO)

Below, `fd_*` functions compute the partial derivative with respect to a single coordinate \( \theta_i \).

You will:
1. Implement **central difference** (2nd order).
2. Implement **4th-order central**.
3. Implement **6th-order central**.

A forward difference implementation is provided as a reference.


In [ ]:

def fd_forward(f, theta, i, h):
    """
    Forward difference (1st order):
        (f(theta + h e_i) - f(theta)) / h
    """
    e = np.zeros_like(theta)
    e[i] = 1.0
    return (f(theta + h*e) - f(theta)) / h

def fd_central_2(f, theta, i, h):
    """
    TODO: Central difference (2nd order):
        (f(theta + h e_i) - f(theta - h e_i)) / (2h)
    """
    # TODO: implement
    raise NotImplementedError("TODO: implement 2nd-order central difference")

def fd_central_4(f, theta, i, h):
    """
    TODO: 4th-order central difference:
        (-f(theta+2h e_i) + 8 f(theta+h e_i) - 8 f(theta-h e_i) + f(theta-2h e_i)) / (12h)
    """
    # TODO: implement
    raise NotImplementedError("TODO: implement 4th-order central difference")

def fd_central_6(f, theta, i, h):
    """
    TODO: 6th-order central difference:
        (f(theta-3h e_i) - 9 f(theta-2h e_i) + 45 f(theta-h e_i)
         - 45 f(theta+h e_i) + 9 f(theta+2h e_i) - f(theta+3h e_i)) / (60h)
    """
    # TODO: implement
    raise NotImplementedError("TODO: implement 6th-order central difference")



## Part 5 — Numerical gradient computation (provided)

The function below computes a numerical gradient vector using a chosen coordinate-wise finite-difference rule.

For speed, you may either:
- compute the full gradient (default, feasible for small models), or
- compute a gradient on a random subset of coordinates (`max_coords`).


In [ ]:

def numerical_grad(f, theta, h, fd_rule, max_coords=None, seed=0):
    """
    Compute numerical gradient of f at theta using fd_rule.

    Parameters
    ----------
    f : callable
        f(theta) -> scalar
    theta : np.ndarray
        shape (m,)
    h : float
        step size
    fd_rule : callable
        fd_rule(f, theta, i, h) -> approx df/dtheta_i
    max_coords : int or None
        if set, randomly sample that many coordinates
    """
    m = theta.size
    g = np.zeros_like(theta)

    if max_coords is None or max_coords >= m:
        coords = np.arange(m)
    else:
        rng = np.random.RandomState(seed)
        coords = rng.choice(m, size=max_coords, replace=False)

    for i in coords:
        g[i] = fd_rule(f, theta, i, h)

    return g, coords

def rel_l2_error(a, b, eps=1e-12):
    return np.linalg.norm(a-b) / (np.linalg.norm(a) + np.linalg.norm(b) + eps)

def linf_error(a, b):
    return np.max(np.abs(a-b))



## Part 6 — Compare finite differences vs backprop (assignment)

You will compare:
- backprop gradient \( \nabla_\theta \mathcal{L} \)
- numerical gradient approximations \( g_{\mathrm{FD}} \)

and report errors as a table across methods and step sizes \( h \).

### Metrics to report

Relative \( \ell_2 \) error:

$$
\frac{\|g_{\mathrm{FD}} - g_{\mathrm{BP}}\|_2}{\|g_{\mathrm{FD}}\|_2 + \|g_{\mathrm{BP}}\|_2}
$$

Infinity norm error:

$$
\|g_{\mathrm{FD}} - g_{\mathrm{BP}}\|_\infty
$$


In [ ]:

# Compute backprop gradient in the same flattened parameter space
params_bp = unpack_params(theta0, meta)
Yhat_bp, cache_bp = forward_mlp(X, params_bp)
loss_bp = mse_loss(Yhat_bp, Y)
grads_bp = backward_mlp(Y, params_bp, cache_bp)

g_bp = np.concatenate([
    grads_bp["dW1"].reshape(-1),
    grads_bp["db1"].reshape(-1),
    grads_bp["dW2"].reshape(-1),
    grads_bp["db2"].reshape(-1),
], axis=0)

print("Loss at theta0:", loss_bp)
print("Backprop grad shape:", g_bp.shape)
print("Backprop grad L2 norm:", np.linalg.norm(g_bp))



### Run gradient check experiments

Start with the forward difference method (already implemented).  
Then implement the TODO methods and rerun this cell.

If runtime is a concern, set `max_coords=200` (or similar) to check a random subset of coordinates.


In [ ]:

def run_experiments(theta0, meta, X, Y, hs, methods, max_coords=None):
    f = lambda th: loss_from_theta(th, meta, X, Y)
    rows = []

    # backprop gradient (full)
    params = unpack_params(theta0, meta)
    Yhat, cache = forward_mlp(X, params)
    grads = backward_mlp(Y, params, cache)
    g_bp = np.concatenate([
        grads["dW1"].reshape(-1),
        grads["db1"].reshape(-1),
        grads["dW2"].reshape(-1),
        grads["db2"].reshape(-1),
    ], axis=0)

    for method_name, fd_rule in methods.items():
        for h in hs:
            try:
                g_fd, coords = numerical_grad(f, theta0, h, fd_rule, max_coords=max_coords, seed=0)

                # If subsampling, compare only those coords for fairness
                if max_coords is not None and max_coords < theta0.size:
                    a = g_fd[coords]
                    b = g_bp[coords]
                else:
                    a = g_fd
                    b = g_bp

                rows.append({
                    "method": method_name,
                    "h": h,
                    "rel_l2_error": rel_l2_error(a, b),
                    "linf_error": linf_error(a, b),
                    "checked_coords": len(coords),
                    "note": "",
                })
            except NotImplementedError as e:
                rows.append({
                    "method": method_name,
                    "h": h,
                    "rel_l2_error": np.nan,
                    "linf_error": np.nan,
                    "checked_coords": max_coords if (max_coords is not None) else theta0.size,
                    "note": str(e),
                })
    return pd.DataFrame(rows)

hs = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]

methods = {
    "forward_1st": fd_forward,
    "central_2nd_TODO": fd_central_2,
    "central_4th_TODO": fd_central_4,
    "central_6th_TODO": fd_central_6,
}

df = run_experiments(theta0, meta, X, Y, hs, methods, max_coords=None)
df



### Interpretation prompt (report writing)

After you implement the TODO methods and rerun:

1. Which method achieves the smallest error for moderate \( h \) (e.g., \(10^{-3}\) to \(10^{-4}\))?
2. Do you observe that **very small** \( h \) (e.g., \(10^{-5}\)) can make error worse? Why?
3. Explain the trade-off between truncation error and floating-point cancellation.



## Part 7 — Grading rubric (100 points)

### A. Understanding + explanation (20 points)
- (10) Clear explanation of why backprop uses the chain rule and reuses intermediate computations
- (10) Correct explanation of why finite differences are used for gradient checking (debugging), not training

### B. Implementation correctness (50 points)
- (10) Correct implementation of central difference (2nd order)
- (15) Correct implementation of 4th-order central difference
- (15) Correct implementation of 6th-order central difference
- (10) Numerical gradient computation runs end-to-end and produces a complete table

### C. Experimental results + table (20 points)
- (10) Table reported for at least 5 values of \( h \) and 3 FD methods
- (10) Correct reporting of relative \( \ell_2 \) and \( \ell_\infty \) error with brief analysis

### D. Short quiz (10 points)
- (10) Answers to quiz questions (below)

---



## Part 8 — Engagement quiz (True/False + short answers)

Answer directly in this notebook (Markdown cell).

### True/False (write T or F and one sentence justification)
1. Backpropagation approximates gradients using finite differences.
2. Central differences are typically more accurate than forward differences for the same step size \( h \).
3. If you always decrease \( h \), finite-difference gradient estimates always become more accurate.
4. The computational cost of finite-difference gradient for \( m \) parameters is \( O(m) \) function evaluations (up to a constant depending on the stencil).
5. Backpropagation computes all partial derivatives in roughly the same order of cost as a few forward passes (up to constants).

### Short answers
6. Why is gradient checking commonly done on a small random subset of parameters in large neural networks?
7. In one paragraph, describe the relationship between MLPs and modern Transformers/LLMs in terms of training (optimization) infrastructure.



## Submission checklist

Before submitting:
- All TODO finite-difference functions are implemented.
- The results table is filled (no NaNs) and discussed.
- Quiz answers are completed.
- Notebook runs top-to-bottom without errors in Google Colab.

---



## Optional extensions (ungraded)

1. Compare numerical gradients against **PyTorch autograd** for the same network.
2. Plot error vs \( h \) for each method on a log scale.
3. Investigate how the best \( h \) changes with float32 vs float64.
